In [ ]:
!pip install -U langchain
!pip install -qU langchain-qdrant
!pip install -U langchain-google-genai
!pip install -U langchain-huggingface
!pip install -qU langchain-openai

In [ ]:
from qdrant_client import QdrantClient
from google.colab import userdata
embedding_model_name = "all-MiniLM-L6-v2"
QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
QDRANT_HOST = userdata.get('QDRANT_HOST') # Ensure this secret's value is just the URL, no quotes.
embedding_model_name = "all-MiniLM-L6-v2"
embedding_dimension = 384
collection_name = "pdf_document_embeddings"

client = QdrantClient(
    url=QDRANT_HOST,
    api_key=QDRANT_API_KEY,
)

### Creating a Langchain Agent with Google Gemini and Qdrant

Now that Qdrant is connected and populated with embeddings, let's create a Langchain agent. This agent will use Google Gemini as its language model and will have access to your Qdrant vector store as a tool to answer questions by retrieving relevant information from your PDF documents.

In [ ]:
import langchain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain_qdrant import QdrantVectorStore
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from qdrant_client.http.models import Distance, VectorParams


In [ ]:

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

embeddings_model = HuggingFaceEmbeddings(model_name=embedding_model_name)
qdrant_vectorstore = QdrantVectorStore(client=client, collection_name=collection_name, embedding=embeddings_model, content_payload_key="text")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:


@tool
def search_qdrant(query: str) -> str:
    """Searches the Qdrant vector store for documents similar to the query."""
    docs = qdrant_vectorstore.similarity_search(query, k=5)
    if docs:
        return docs[0].page_content
    return "No relevant documents found in Qdrant."

tools = [search_qdrant]
system_message_content = "You are a helpful assistant. Use the available tools to answer questions about the provided documents. If the question can be answered by searching Qdrant, use the 'search_qdrant' tool."

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message_content),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

graph = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message_content,
)

In [ ]:
question = "what is the invoice number"
question = "what of the invoice have cheaper same items then other vendors"

In [ ]:
inputs = {"messages": [{"role": "user", "content": question}]}


for chunk in graph.stream(inputs, stream_mode="updates"):
    print(chunk)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_qdrant', 'arguments': '{"query": "invoice cheaper items other vendors"}'}, '__gemini_function_call_thought_signatures__': {'ee6f01ca-1dc7-46e0-8436-4a9538523b04': 'Ct0CARFNMg984YqS7dzkFYkd38U6msy4DwVMi0Tc/pnuzpfWgODZZyuKafRkAjaVym6yiPhb14mLv/UYHIkBTpZXRTIPp40qG2tGeo2aFW6QC5YLtpSFAJzH24Qu2a3ej7oTIt/z5mLZ/2IVo5P5xwwOPKaNkzY9fV7X/f3kyI3CC7c5UKJYpg4CkRWZiXaGRuRGchUAYnr8umez1GkhyioX2bJlgo2BJ90u8wCxQa4L5DL52QvcTKYnyYOVxrEASaV9jz/u1OOe38nYsDk63VuRF/regA8NNSQhpX50vchdD0+TZgCqeilg46OXKm42iq3Vs/uYQkWXAF7o73QXMWcpZ8A8qxiTFyXVxcYU5NjnZ6J7l8DDDVD4kBst9vRhR10DmZyn8cG7u+ZWCuNwmA59vE1qW+9DRyQequXhI9Pb9Y032pA2eFZVGgF3k2sh2TC3VVvqZbvU/9q1A85xcg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fefc1-7a32-7112-9a3b-55675d95d3cb-0', tool_calls=[{'name': 'search_qdrant', 'args': {'query': 'invoice cheaper

In [ ]:
from langchain_openai import ChatOpenAI
from google.colab import userdata

MODEL_NAME = userdata.get('MODEL_NAME')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
OPENAI_API_LINK = userdata.get('OPENAI_API_LINK')
llm= ChatOpenAI(
   model_name=MODEL_NAME,
   openai_api_key=OPENAI_API_KEY,
   openai_api_base=OPENAI_API_LINK
)

print(f"OpenAI LLM '{MODEL_NAME}' initialized successfully.")

OpenAI LLM 'gpt-5.4-nano' initialized successfully.


In [ ]:
graph2 = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_message_content,
)

In [ ]:
inputs = {"messages": [{"role": "user", "content": question}]}


for chunk in graph2.stream(inputs, stream_mode="updates"):
    print(chunk)

final_message = chunk.get('model', {}).get('messages', [None])[0]

if final_message:
    if isinstance(final_message.content, str):
        print(final_message.content)
    elif isinstance(final_message.content, list):
        extracted_text = ""
        for content_block in final_message.content:
            if isinstance(content_block, dict) and 'text' in content_block:
                extracted_text += content_block['text']
            elif isinstance(content_block, str):
                extracted_text += content_block
        print(extracted_text)
    else:
        print("Final message content has an unexpected type.")
else:
    print("No final message found in the last chunk.")

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 188, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EBbLRxaZXyzDPn8hjp8AhxruGRJ4F', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fefc1-927b-7520-9871-947ed4ade7f3-0', tool_calls=[{'name': 'search_qdrant', 'args': {'query': 'cheaper same items than other vendors invoice'}, 'id': 'call_v8IRJmOWnU45lRikGBVvB0Vy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 26, 'total_tokens': 214, 'input_token_details': {'audio': 0,